In [ ]:
import sys
!{sys.executable} -m pip install plotly kaleido open3d

In [1]:
import os
import glob
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

IMG_PATH: str = "img"

def check_folder_exists(folder_path: str) -> bool:
    return os.path.exists(folder_path)

def create_folder(folder_path: str) -> None:
    os.makedirs(folder_path, exist_ok=True)


def load_pairs(folder):

    gt_files = glob.glob(os.path.join(folder, "*_gt.pcd"))

    pairs = []

    for gt_path in gt_files:

        prefix = os.path.basename(gt_path).replace("_gt.pcd", "")

        partial_path = os.path.join(folder, f"{prefix}_partial.pcd")

        if check_folder_exists(partial_path):
            pairs.append((prefix, gt_path, partial_path))

    return pairs


def center_points(points):

    center = points.mean(axis=0)
    return points - center


def visualize_pair_plotly(name,name_model, gt_path, partial_path):

    print(f"\nVisualizing: {name}")

    gt_pcd = o3d.io.read_point_cloud(gt_path)
    partial_pcd = o3d.io.read_point_cloud(partial_path)

    gt_points = np.asarray(gt_pcd.points)
    partial_points = np.asarray(partial_pcd.points)

    # Zentrieren
    gt_points = center_points(gt_points)
    partial_points = center_points(partial_points)

    # Abstand bestimmen
    max_extent = np.max(gt_points.max(axis=0) - gt_points.min(axis=0))
    shift_x = max_extent * 1.5

    # Partial nach rechts
    partial_points[:, 0] += shift_x

    fig = go.Figure()

    # GT
    fig.add_trace(
        go.Scatter3d(
            x=gt_points[:, 0],
            y=gt_points[:, 1],
            z=gt_points[:, 2],
            mode='markers',
            marker=dict(
                size=2,
                color="black"
            ),
            name='GT'
        )
    )

    # PARTIAL
    fig.add_trace(
        go.Scatter3d(
            x=partial_points[:, 0],
            y=partial_points[:, 1],
            z=partial_points[:, 2],
            mode='markers',
            marker=dict(
                size=2,
                color="orange"
            ),
            name='PARTIAL'
        )
    )

    fig.update_layout(
        title=f"{name}_{name_model}",
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        height=900,
        showlegend=True
    )

    fig.show()

    #fig.write_image(f"{IMG_PATH}/{name}_{name_model}.svg")


def visualize_main(dataset_dir: str):

    pairs = load_pairs(dataset_dir)
    name_model = os.path.basename(dataset_dir)
    print(f" {name_model}")

    print(f"{len(pairs)} pairs found")

    if not check_folder_exists(IMG_PATH):
        create_folder(IMG_PATH)


    for name, gt_path, partial_path in pairs:
        visualize_pair_plotly(name, name_model, gt_path, partial_path)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
visualize_main("SnowFlakeNet")


 SnowFlakeNet
6 pairs found

Visualizing: best_0_hung_0_f55345cf-63ec-4ed2-be4c-3b27b80eda70_cdl1_2.29



Visualizing: best_1_hung_4_0a7ba2cf-f199-449a-94e7-6d8d636f1e21_cdl1_2.36



Visualizing: best_2_hung_2_fa7d49c8-317c-44f2-87e4-c5ff79121f7d_cdl1_2.37



Visualizing: worst_0_dutch_0_5a251d14-201e-4812-b9db-9b380443592b_cdl1_23.37



Visualizing: worst_1_dutch_1_7bb61e49-9cc6-429f-925f-d87085976c49_cdl1_24.55



Visualizing: worst_2_dutch_1_4cc36566-1e10-4a21-9e3d-9a611caa1e69_cdl1_30.16


In [3]:
visualize_main("TopNet")

 TopNet
6 pairs found

Visualizing: best_0_sncf_0_b6c6804f-5def-4c47-a324-546343c54fdb_cdl1_24.16



Visualizing: best_1_sncf_0_61bc5556-4cea-410f-8c68-07160b394ca0_cdl1_24.32



Visualizing: best_2_chine_2_46510298-86f0-4000-a5a5-96fe1b4b912e_cdl1_26.06



Visualizing: worst_0_hung_3_e5a035ce-8d01-4427-87f1-e8d042076b09_cdl1_207.43



Visualizing: worst_1_dutch_4_b3186ecc-baf2-4e61-920f-c4d835545bad_cdl1_209.13



Visualizing: worst_2_dutch_1_7bb61e49-9cc6-429f-925f-d87085976c49_cdl1_394.14
